# **Code of the Practice Session**

In [12]:
import random
from copy import deepcopy
import numpy as np
import matplotlib.pyplot as plt

In [13]:
def reward_function(state):
  """
    Reward function takes a state and return
    the possible rewards in that state.
    Note that the reward function is usually given as
    a function of (s, a, s') but here, the rewards only
    depend on the state you land in no matter which action
    you took and which state you came from

  """
  if state=="A":
    return [-1, 0]
  elif state=="B":
    return [0]
  elif state=="C":
    return [0]
  elif state=="D":
    return [1]
  else:
    raise f"state {state} does not exist in state space: {state_space}"

def atree(next_state, action, reward, policy_dist):
  """
    This computes the joint probability of a transitioning
    from state A to any next_state and receiving the reward.
    It's basically the tree of A that we saw in Fig.1.b
  """
  if next_state=="A" and action=="left":
    if reward==0:
      return 0.3*1*policy_dist[action]
    elif reward==-1:
      return 0.7*1*policy_dist[action]
    else :
      return 0

  elif next_state=="B" and action=="right":
    if reward==0:
      return 1*1*policy_dist[action]
    else:
      return 0
  else:
    return 0

def btree(next_state, action, reward, policy_dist):
  """
    This computes the joint probability of a transitioning
    from state B to any next_state and receiving the reward.
    It's basically the tree of B that we saw in Fig.1.b
  """
  if next_state=="A" and action=="left":
    if reward==0:
      return 0.3*1*policy_dist[action]
    elif reward==-1:
      return 0.7*1*policy_dist[action]
    else:
      return 0

  elif next_state=="C" and action=="right":
    if reward==0:
      return 1*1*policy_dist[action]
    else:
      return 0
  else:
    return 0

def ctree(next_state, action, reward, policy_dist):
  """
    This computes the joint probability of a transitioning
    from state C to any next_state and receiving the reward.
    It's basically the tree of C that we saw in Fig.1.b
  """
  if next_state=="A" and action=="left":
    if reward==0:
      return 0.3*0.5*policy_dist[action]
    elif reward==-1:
      return 0.7*0.5*policy_dist[action]
    else:
      return 0
  elif next_state=="B" and action=="left":
    if reward==0:
      return 1.*0.5*policy_dist[action]
    else:
      return 0

  elif next_state=="D" and action=="right":
    if reward==1:
      return 1*1*policy_dist[action]
    else:
      return 0
  else:
    return 0

def dtree(next_state, action, reward, policy_dist):
  """
    No transition from D because it's the final state. Thus probability is 0
  """

  return 0


def transition_probability_function(next_state, reward, state, action, policy_dist):
  """
    Full dynamic of this Markov Decision Process (MDP)
  """
  if state=="A":
     prob = atree(next_state, action, reward, policy_dist)
  elif state=="B":
    prob = btree(next_state, action, reward, policy_dist)
  elif state=="C":
    prob = ctree(next_state, action, reward, policy_dist)
  elif state=="D":
    prob = dtree(next_state, action, reward, policy_dist)
  else:
    raise f"state {state} does not exist in state space: {state_space}"

  return prob

In [14]:
def test(state_space, policy_dist):
  """
    This is a small test to unsure every transition
    sump up to 1 for each state (except D) and action.
    Remember that it's a distribution.
  """
  for state in state_space:
    prob = 0
    for action in policy_dist.keys():
      for next_state in state_space:
        for reward in reward_function(next_state):
          prob += transition_probability_function(next_state, reward, state, action, policy_dist)
    if state=="D":
      assert prob==0, "No transition from D, it's prob must be 0"
      print(f"state {state}: OK")
    else:
      assert prob>=0.99 and prob<=1.01, f"All transition probabilities must sum up to 1 for each s and a,\
                      you have prob: {prob} for state {state}"
      print(f"state {state}, prob: {prob}: OK")


In [67]:
NUM_ITER = 10
gamma = 0.8
delta = 1.
iter = 0

state_space = ["A", "B", "C", "D"]
action_space = ["left", "right"]
policy_distribution = {"left":0.3, "right":0.7}
state_value_dict = {"A":0, "B":0, "C":0, "D":0} # a scalar for each state
previous_state_value_dict = deepcopy(state_value_dict)


while delta>1e-6:
  iter += 1
  for state in state_space:
    value = 0 # initliaze the value of the state
    for action in action_space:
      temp_sum = 0

      # update the policy distribution into a deterministic distribution. 1 for the current action and 0 for the other
      # This is because our tranisition rewards depend on the policy distribution

      for act in policy_distribution:
        if act==action:
          policy_distribution[act] = 1.0
        else:
          policy_distribution[act] = 0.0

      for next_state in state_space:
        possible_rewards = reward_function(next_state)
        for reward_obtained in possible_rewards:
          next_state_value = previous_state_value_dict[next_state]
          temp_sum += transition_probability_function(next_state, reward_obtained, state, action, policy_distribution)*(reward_obtained + gamma*next_state_value)

      value = temp_sum if value <= temp_sum else value

    state_value_dict[state] = round(value, 3)


  delta = np.linalg.norm(np.array(list(previous_state_value_dict.values())) - np.array(list(state_value_dict.values())))
  print(f"INFO Iteration: {iter}, value of each state is: {state_value_dict} delta: {delta}")
  previous_state_value_dict = deepcopy(state_value_dict)

INFO Iteration: 1, value of each state is: {'A': 0.0, 'B': 0.0, 'C': 1.0, 'D': 0.0} delta: 1.0
INFO Iteration: 2, value of each state is: {'A': 0.0, 'B': 0.8, 'C': 1.0, 'D': 0.0} delta: 0.8
INFO Iteration: 3, value of each state is: {'A': 0.64, 'B': 0.8, 'C': 1.0, 'D': 0.0} delta: 0.64
INFO Iteration: 4, value of each state is: {'A': 0.64, 'B': 0.8, 'C': 1.0, 'D': 0.0} delta: 0.0


In [65]:
# This is just a snippet of code of policy evaluation, the full code is on the github
state_space = ["A", "B", "C", "D"]
action_space = ["left", "right"]
policy_distribution = {"left":0., "right":1.0}
state_value_dict = {"A":0, "B":0, "C":0, "D":0} # a scalar for each state
reward_space = {"A": [-1, 0], "B":[0], "C":[0], "D":[1]}
test(state_space, policy_distribution)

# this will hold the old value function
# when we are computing the new value function. This is called the two arrays
# method in Sutton & Barto. But you can do with only one value function also
# which is called the in place method
# So with two arrays, you have: value(s) = p(s', r|a, s)*[r + old value(s')]
# And with in-place update, you have: value(s) = p(s', r|a, s)*[r + value(s')]

# You are free to use whatever method you want, though the in-place update converges
# faster than the two-arrays version because it immediately uses the value of
# the state s as long as it become available. whereas with two-arrays, the value
# array remains fixed for ONE FULL ITERATION !

# But... we will use the two-arrays version here !!!!!

NUM_ITER = 10
gamma = 0.8

previous_state_value_dict = deepcopy(state_value_dict)

for iter in range(NUM_ITER):
  # One iteration visits all states at least once
  for state in state_space:
    value = 0 # initliaze the value of the state
    for action in action_space:
      temp_sum = 0
      for next_state in state_space: # all state are considered possible
        #next state. Some probabilities are zero ofc when no transition exists
        possible_rewards = reward_space[next_state]
        for reward_obtained in possible_rewards:
          next_state_value = previous_state_value_dict[next_state]
          temp_sum += transition_probability_function(next_state, reward_obtained, state, action, policy_distribution)*(reward_obtained + gamma*next_state_value)
      value += policy_distribution[action]*temp_sum

    state_value_dict[state] = round(value, 3)


  print(f"INFO Iteration: {iter}, value of each state is: {state_value_dict}")
  previous_state_value_dict = deepcopy(state_value_dict)

state A, prob: 1.0: OK
state B, prob: 1.0: OK
state C, prob: 1.0: OK
state D: OK
INFO Iteration: 0, value of each state is: {'A': 0.0, 'B': 0.0, 'C': 1.0, 'D': 0.0}
INFO Iteration: 1, value of each state is: {'A': 0.0, 'B': 0.8, 'C': 1.0, 'D': 0.0}
INFO Iteration: 2, value of each state is: {'A': 0.64, 'B': 0.8, 'C': 1.0, 'D': 0.0}
INFO Iteration: 3, value of each state is: {'A': 0.64, 'B': 0.8, 'C': 1.0, 'D': 0.0}
INFO Iteration: 4, value of each state is: {'A': 0.64, 'B': 0.8, 'C': 1.0, 'D': 0.0}
INFO Iteration: 5, value of each state is: {'A': 0.64, 'B': 0.8, 'C': 1.0, 'D': 0.0}
INFO Iteration: 6, value of each state is: {'A': 0.64, 'B': 0.8, 'C': 1.0, 'D': 0.0}
INFO Iteration: 7, value of each state is: {'A': 0.64, 'B': 0.8, 'C': 1.0, 'D': 0.0}
INFO Iteration: 8, value of each state is: {'A': 0.64, 'B': 0.8, 'C': 1.0, 'D': 0.0}
INFO Iteration: 9, value of each state is: {'A': 0.64, 'B': 0.8, 'C': 1.0, 'D': 0.0}
